# broadcast-source-fanout — faded example 3: Fill the loss whose gradient counts codebook fan-out usage

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-source-fanout`. The last cell reports your progress on the `Generative: Broadcast source fan-out` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Broadcast source fan-out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-source-fanout`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-source-fanout"
DD_SUBTOPIC = "Generative: Broadcast source fan-out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Indexing a leaf `(K, D)` codebook by code ids fans rows out to `(B, L, D)`. If you reduce the fan-out with `sum()` and backprop, each codebook row's gradient becomes a constant vector equal to the number of times that row was selected; absent rows stay exactly zero. The blanked step is the scalar reduction that gives this clean per-element gradient of 1.

## Faded exercise 3

Complete `codebook_usage_grad(K, D, codes)`. It builds a leaf codebook, fans it out via `codebook[codes]`, then must form a scalar loss whose backward routes a gradient of exactly 1 to every fanned-out element (so each codebook row's gradient counts its usage). Fill in that scalar-loss step; the backward call and grad return are given.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
from torch import Tensor

def codebook_usage_grad(K: int, D: int, codes: Tensor):
    codebook = t.arange(K * D, dtype=t.float32).reshape(K, D).requires_grad_(True)
    fanned = codebook[codes]
    loss = None  # TODO: fill in this step — read the prompt cell above
    loss.backward()
    return codebook.grad.clone()


def _test():
    import torch as t
    codes = t.tensor([[0, 0, 2], [2, 2, 0]])
    grad = codebook_usage_grad(4, 3, codes)
    assert tuple(grad.shape) == (4, 3)
    # id 0 appears 3x, id 2 appears 3x, ids 1 & 3 absent
    import collections
    counts = collections.Counter(codes.flatten().tolist())
    for k in range(4):
        expected = float(counts.get(k, 0))
        assert t.allclose(grad[k], t.full((3,), expected)), (k, grad[k])


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
from torch import Tensor

def codebook_usage_grad(K: int, D: int, codes: Tensor):
    codebook = t.arange(K * D, dtype=t.float32).reshape(K, D).requires_grad_(True)
    fanned = codebook[codes]
    loss = fanned.sum()
    loss.backward()
    return codebook.grad.clone()
```
</details>